# JPSS (Suomi NPP, NOAA-20, NOAA-21) / VIIRS (Visible Infrared Imaging Radiometer Suite)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=noaa/jpss-viirs). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1B

Level 1B (NASA VNP02/VJ102/VJ202): calibrated radiances and reflectances for the M (VNP02MOD), I (VNP02IMG) bands and DNB, with geolocation in VNP03.

Searches NASA CMR for one VIIRS L1B granule (Suomi NPP, VNP02MOD) over Rome, downloads it with an Earthdata Login and plots M15. Use VJ102MOD for NOAA-20 and VJ202MOD for NOAA-21.

In [ ]:
%pip install -q earthaccess xarray netCDF4 h5netcdf matplotlib

In [ ]:
import earthaccess
import matplotlib.pyplot as plt
import xarray as xr

earthaccess.login()                              # NASA Earthdata Login (free account), interactive
results = earthaccess.search_data(short_name="VNP02MOD", temporal=("2024-06-01", "2024-06-01"),
                                  bounding_box=(12.0, 41.0, 13.0, 42.5), count=1)
print(len(results), "granule(s)")
files = earthaccess.download(results, "data")

ds = xr.open_dataset(files[0], group="observation_data")
print(list(ds.data_vars)[:30])
da = ds["M15"]                                   # brightness temperature, 10.76 um (K)
da[::4, ::4].plot(cmap="gray_r", robust=True)
plt.title("VIIRS M15 brightness temperature")
plt.show()

## L2

Level 2: geophysical products (NASA VNP09 surface reflectance, VNP14 thermal anomalies/fire, VNP21 land surface temperature, VNP35 cloud mask) and NOAA EDRs (e.g. JRR cloud/aerosol/fire).

Searches NASA CMR for one VIIRS thermal anomalies (fire) granule over Rome, downloads it with an Earthdata Login and plots the fire mask (or the first 2D variable).

In [ ]:
%pip install -q earthaccess xarray netCDF4 h5netcdf matplotlib

In [ ]:
import earthaccess
import matplotlib.pyplot as plt
import xarray as xr

earthaccess.login()                              # NASA Earthdata Login (free account), interactive
results = earthaccess.search_data(short_name="VNP14", temporal=("2024-06-01", "2024-06-01"),
                                  bounding_box=(12.0, 41.0, 13.0, 42.5), count=1)
print(len(results), "granule(s)")
files = earthaccess.download(results, "data")

ds = xr.open_dataset(files[0])
print(ds)
two_d = [n for n, v in ds.data_vars.items() if v.ndim == 2]
print(two_d)
ds["fire mask" if "fire mask" in ds else two_d[0]].plot()
plt.show()

## L3

Level 3: gridded NASA products such as VNP13A1 vegetation indices (500 m, 16-day), VNP43 BRDF/albedo, VNP15 LAI/FPAR and VNP46 black marble night lights.

Annual NDVI time series at a point near Rome from the VIIRS VNP13A1 collection in Google Earth Engine (needs a Google Cloud project with Earth Engine enabled).

In [ ]:
%pip install -q earthengine-api pandas matplotlib

In [ ]:
import ee
import matplotlib.pyplot as plt
import pandas as pd

ee.Authenticate()                                # opens a browser/Colab login the first time
PROJECT = input("Google Cloud project ID registered for Earth Engine: ")
ee.Initialize(project=PROJECT)

point = ee.Geometry.Point([12.5, 41.9])
ic = ee.ImageCollection("NASA/VIIRS/002/VNP13A1").filterDate("2023-01-01", "2023-12-31").select("NDVI")
feats = ic.map(lambda im: ee.Feature(None, {"date": im.date().format("YYYY-MM-dd"),
                                            "ndvi": im.reduceRegion(ee.Reducer.mean(), point, 500).get("NDVI")}))
rows = [f["properties"] for f in feats.getInfo()["features"]]
df = pd.DataFrame(rows).dropna()
df["date"] = pd.to_datetime(df["date"])
df["ndvi"] = df["ndvi"] * 0.0001                 # scale factor
df.set_index("date")["ndvi"].plot(marker="o")
plt.title("VIIRS VNP13A1 NDVI, Rome, 2023")
plt.show()